# ASRS fan-out / fan-in demo for flight-delay analysis

This notebook shows a practical **fan-out / fan-in** LLM workflow using narrative aviation safety reports.

## Why this is useful

Your sample file contains many ASRS-style reports with free-text narratives plus structured metadata. A good LLM workflow for this kind of data is:

1. **Parse reports** from the text file.
2. **Fan out** each report into several focused LLM tasks that run concurrently.
3. **Fan in** the intermediate outputs into one structured synthesis for each report.
4. Optionally do a **second fan-in** across all synthesized reports to produce a portfolio-level summary of likely delay drivers.

## Fan-out plan

For each report, run these LLM calls in parallel:

- **Event extractor**: what happened, where, and during what phase?
- **Delay driver classifier**: which delay-causing categories are present?
- **Operational impact assessor**: did this likely cause delay, cancellation, go-around, gate return, tow, inspection, or congestion?
- **Mitigation analyst**: what procedural, airport, maintenance, or ATC fixes are suggested?

Then combine those outputs into a single normalized JSON record.

## Why this fits your project

This lets you connect **narrative operational reports** to the kinds of delay causes you care about in BTS flight data:

- weather
- airport surface congestion / layout
- ATC / airspace
- ground handling / ramp
- maintenance / aircraft systems
- crew communication / situational awareness

In [1]:
from __future__ import annotations

import asyncio
import json
import os
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd
from openai import AsyncOpenAI

In [3]:
DATA_PATH = Path("airline_reports.txt")
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
MAX_REPORTS = 8          # keep the demo small and fast
MAX_CHARS_PER_REPORT = 4000
CONCURRENCY_LIMIT = 8    # global limit across all LLM calls

assert DATA_PATH.exists(), f"Could not find {DATA_PATH}"
print(DATA_PATH, "exists")

airline_reports.txt exists


## Parse the raw text into individual reports

Each report starts with an `ACN:` line. We will split on that boundary and keep the narrative-heavy text for LLM analysis.

In [4]:
raw_text = DATA_PATH.read_text(encoding="utf-8", errors="ignore")

# Split while keeping the ACN marker as part of each chunk
parts = re.split(r"(?=ACN:\s*\d+)", raw_text)
parts = [p.strip() for p in parts if p.strip()]

print(f"Parsed {len(parts)} report chunks")
print(parts[0][:1000])

Parsed 37 report chunks
ACN: 2305728
Time / Day
Date : 202510
Local Time Of Day : 0601-1200
Place
Locale Reference.Airport : BCV.Airport
State Reference : AK
Altitude.MSL.Single Value : 1083
Environment
Flight Conditions : VMC
Weather Elements / Visibility.Visibility : 10
Light : Daylight
Ceiling.Single Value : 9000
Aircraft : 1
Reference : X
ATC / Advisory.CTAF : BCV
Aircraft Operator : Personal
Make Model Name : Small Aircraft, High Wing, 1 Eng, Fixed Gear
Operating Under FAR Part : Part 91
Flight Plan : None
Mission : Personal
Flight Phase : Taxi
Flight Phase : Takeoff / Launch
Airspace.Class G : BCV
Aircraft : 2
Reference : Y
ATC / Advisory.CTAF : BCV
Make Model Name : Small Aircraft, High Wing, 1 Eng, Fixed Gear
Crew Size.Number Of Crew : 1
Flight Phase : Takeoff / Launch
Airspace.Class G : BCV
Person
Location Of Person.Aircraft : X
Location In Aircraft : Flight Deck
Reporter Organization : Personal
Function.Flight Crew : Single Pilot
Qualification.Flight Crew : Commercial
Qualifi

In [5]:
def extract_field(pattern: str, text: str, default: str | None = None) -> str | None:
    m = re.search(pattern, text, flags=re.IGNORECASE | re.MULTILINE)
    return m.group(1).strip() if m else default


def extract_narratives(text: str) -> list[str]:
    # Captures "Narrative: 1" blocks until the next "Narrative:", "Synopsis", or end
    pattern = r"Narrative:\s*\d+\s*(.*?)(?=\n(?:Narrative:\s*\d+|Synopsis)\b|\Z)"
    return [m.strip() for m in re.findall(pattern, text, flags=re.DOTALL | re.IGNORECASE)]


def make_report_record(text: str) -> dict[str, Any]:
    acn = extract_field(r"ACN:\s*(\d+)", text)
    date_yyyymm = extract_field(r"Date\s*:\s*(\d+)", text)
    airport = extract_field(r"Locale Reference\.Airport\s*:\s*([A-Z0-9]+)\.Airport", text)
    state = extract_field(r"State Reference\s*:\s*([A-Z]{2}|US|DC|ON|AK|PA|NM|AR|MT|KY|TN|NV|MO|GA|CA|SC|CO)", text)
    primary_problem = extract_field(r"Primary Problem\s*:\s*(.+)", text)
    synopsis = extract_field(r"Synopsis\s*(.*)", text)
    narratives = extract_narratives(text)
    narrative_text = "\n\n".join(narratives)
    if len(narrative_text) > MAX_CHARS_PER_REPORT:
        narrative_text = narrative_text[:MAX_CHARS_PER_REPORT] + "\\n...[truncated for demo]"

    return {
        "acn": acn,
        "date_yyyymm": date_yyyymm,
        "airport": airport,
        "state": state,
        "primary_problem": primary_problem,
        "synopsis": synopsis,
        "narratives": narratives,
        "report_text": text[:MAX_CHARS_PER_REPORT],
        "analysis_text": narrative_text if narrative_text else text[:MAX_CHARS_PER_REPORT],
    }


reports = [make_report_record(p) for p in parts]
df_reports = pd.DataFrame(reports)
df_reports.head(10)

,acn,date_yyyymm,airport,state,primary_problem,synopsis,narratives,report_text,analysis_text
0,2305728,202510,BCV,AK,Airport,General aviation pilot reported PABV/BCV Airpo...,[This is the nth report for many of the same t...,ACN: 2305728\nTime / Day\nDate : 202510\nLocal...,This is the nth report for many of the same th...
1,2305051,202511,ZZZ,US,Human Factors,B737 flight crew and ramp agent reported the a...,[Pushback commenced normally with a standard s...,ACN: 2305051\nTime / Day\nDate : 202511\nLocal...,Pushback commenced normally with a standard se...
2,2303590,202511,ZZZ,US,Human Factors,Pilot reported a taxiing aircraft collided wit...,[Flight from ZZZ1 to ZZZ airport under day VFR...,ACN: 2303590\nTime / Day\nDate : 202511\nLocal...,Flight from ZZZ1 to ZZZ airport under day VFR....
3,2301677,202511,ZZZ,US,Aircraft,Cessna 208 First Officer reported a brake malf...,"[On our final flight back to ZZZ Airport, an i...",ACN: 2301677\nTime / Day\nDate : 202511\nLocal...,"On our final flight back to ZZZ Airport, an in..."
4,2301503,202510,PHL,PA,Procedure,Regional jet captain reported declaring minimu...,[Departing ZZZ reaching cruise we noticed the ...,ACN: 2301503\nTime / Day\nDate : 202510\nLocal...,Departing ZZZ reaching cruise we noticed the t...
5,2299197,202510,ABQ,NM,Chart Or Publication,Air carrier pilot reported the ABQ 10-7 pages ...,[Regarding ramp taxi in instructions. Landed R...,ACN: 2299197\nTime / Day\nDate : 202510\nLocal...,Regarding ramp taxi in instructions. Landed Ru...
6,2298390,202510,LIT,AR,Ambiguous,Air taxi Captain reported a taxi deviation ont...,[Revenue flight from LIT to ZZZ. Our assigned ...,ACN: 2298390\nTime / Day\nDate : 202510\nLocal...,Revenue flight from LIT to ZZZ. Our assigned d...
7,2294724,202510,S27,MT,Human Factors,General aviation Flight Instructor reported a ...,"[Events occured at s27, an untowered airport w...",ACN: 2294724\nTime / Day\nDate : 202510\nLocal...,"Events occured at s27, an untowered airport wi..."
8,2290406,202509,CVG,KY,Procedure,Air carrier Captain reported CVG Ramp Controll...,[Upon switching from Ground Control to Ramp To...,ACN: 2290406\nTime / Day\nDate : 202509\nLocal...,Upon switching from Ground Control to Ramp Tow...
9,2289681,202509,CYYZ,ON,Airport,Air carrier Captain reported a safety concern ...,[Less than 10 feet from the stop line at gate ...,ACN: 2289681\nTime / Day\nDate : 202509\nLocal...,Less than 10 feet from the stop line at gate X...


## Choose a working subset for the demo

The file contains many reports. For a notebook demo, it is usually better to analyze a small batch so the fan-out pattern is visible and affordable.

In [6]:
demo_df = (
    df_reports.loc[df_reports["analysis_text"].str.len() > 200]
    .head(MAX_REPORTS)
    .copy()
)

print(f"Using {len(demo_df)} reports for the demo")
demo_df[["acn", "date_yyyymm", "airport", "state", "primary_problem", "synopsis"]]

Using 8 reports for the demo


,acn,date_yyyymm,airport,state,primary_problem,synopsis
0,2305728,202510,BCV,AK,Airport,General aviation pilot reported PABV/BCV Airpo...
1,2305051,202511,ZZZ,US,Human Factors,B737 flight crew and ramp agent reported the a...
2,2303590,202511,ZZZ,US,Human Factors,Pilot reported a taxiing aircraft collided wit...
3,2301677,202511,ZZZ,US,Aircraft,Cessna 208 First Officer reported a brake malf...
4,2301503,202510,PHL,PA,Procedure,Regional jet captain reported declaring minimu...
5,2299197,202510,ABQ,NM,Chart Or Publication,Air carrier pilot reported the ABQ 10-7 pages ...
6,2298390,202510,LIT,AR,Ambiguous,Air taxi Captain reported a taxi deviation ont...
7,2294724,202510,S27,MT,Human Factors,General aviation Flight Instructor reported a ...


## Define the four fan-out tasks

Each task is narrow and explicit, which helps produce cleaner results than asking one giant prompt to do everything at once.

In [7]:
SYSTEM_PROMPT = """
You are an aviation operations analyst helping connect ASRS narrative reports to airline delay and disruption drivers.
Return strict JSON only. Do not wrap in markdown fences.
"""

TASK_SPECS = {
    "event_extractor": """
Extract a compact event summary from this report.

Return JSON with keys:
- event_summary: one sentence
- flight_phase
- airport
- operation_type
- immediate_outcome
- evidence_quote: a short supporting quote or phrase
""",
    "delay_driver_classifier": """
Classify likely delay/disruption drivers present in this report.

Return JSON with keys:
- primary_driver: one of [weather, airport_layout, airport_surface_congestion, atc_airspace, ground_ops, maintenance_aircraft, crew_human_factors, chart_publication, security_other]
- secondary_drivers: list of zero or more from the same set
- likely_delay_mechanism: one sentence explaining how this could create delay or disruption
- confidence: integer 1-5
""",
    "operational_impact_assessor": """
Assess likely operational impact to schedule or movement.

Return JSON with keys:
- likely_delay: true/false
- likely_cancellation: true/false
- likely_gate_return: true/false
- likely_go_around_or_resequence: true/false
- disruption_severity: integer 1-5
- impact_types: list chosen from [taxi_delay, pushback_delay, departure_delay, arrival_delay, gate_return, aircraft_swap, cancellation, go_around, runway_blockage, inspection, towing, passenger_impact, none]
- rationale: short explanation
""",
    "mitigation_analyst": """
Identify the most useful mitigations.

Return JSON with keys:
- mitigation_bucket: one of [procedure, training, signage_markings, airport_infrastructure, maintenance, dispatch_planning, atc_phraseology, ground_crew_coordination, technology]
- recommended_actions: list of 3 concise actions
- who_should_act: list chosen from [airline, airport, atc, maintenance, dispatch, ramp_ops, pilots, faa]
"""
}

## Async helpers

This is where the **fan-out** pattern shows up. For each report, the notebook launches multiple LLM calls concurrently with `asyncio.gather(...)`. A semaphore limits total concurrency.

In [8]:
client = AsyncOpenAI()
semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)


async def run_json_task(task_name: str, report: dict[str, Any]) -> dict[str, Any]:
    prompt = f"""
Task: {task_name}

Instructions:
{TASK_SPECS[task_name]}

Context fields:
ACN: {report.get("acn")}
Date: {report.get("date_yyyymm")}
Airport: {report.get("airport")}
Primary problem: {report.get("primary_problem")}
Synopsis: {report.get("synopsis")}

Narrative text:
{report.get("analysis_text")}
""".strip()

    async with semaphore:
        response = await client.responses.create(
            model=MODEL,
            instructions=SYSTEM_PROMPT,
            input=prompt,
            temperature=0,
        )

    text = response.output_text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # Make notebook debugging easier if the model returns malformed JSON
        return {"_parse_error": True, "_raw_text": text, "task_name": task_name}


async def fan_out_report(report: dict[str, Any]) -> dict[str, Any]:
    task_names = list(TASK_SPECS.keys())
    results = await asyncio.gather(*(run_json_task(name, report) for name in task_names))
    return dict(zip(task_names, results))

## Fan-in synthesis for each report

Once the parallel tasks finish, we perform a single synthesis call that merges the partial outputs into one normalized record. This is the **fan-in** step for each report.

In [9]:
SYNTHESIS_SYSTEM = """
You are merging partial aviation-analysis outputs into one normalized record.
Return strict JSON only.
"""

async def synthesize_report(report: dict[str, Any], fanout_results: dict[str, Any]) -> dict[str, Any]:
    prompt = f"""
Merge the following partial analyses into one report-level JSON record.

Original context:
ACN: {report.get("acn")}
Date: {report.get("date_yyyymm")}
Airport: {report.get("airport")}
State: {report.get("state")}
Primary problem: {report.get("primary_problem")}
Synopsis: {report.get("synopsis")}

Partial analyses:
{json.dumps(fanout_results, indent=2)}

Return JSON with keys:
- acn
- airport
- date_yyyymm
- concise_summary
- primary_driver
- secondary_drivers
- likely_delay
- likely_cancellation
- likely_gate_return
- likely_go_around_or_resequence
- disruption_severity
- impact_types
- mitigation_bucket
- recommended_actions
- analyst_notes
"""

    async with semaphore:
        response = await client.responses.create(
            model=MODEL,
            instructions=SYNTHESIS_SYSTEM,
            input=prompt,
            temperature=0,
        )

    text = response.output_text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {
            "acn": report.get("acn"),
            "_parse_error": True,
            "_raw_text": text,
        }

## End-to-end pipeline for one batch

For each report:

- fan out to four concurrent LLM calls
- fan in to one report-level synthesis

This gives you a clean structure you can later join to operational data.

In [10]:
async def process_one_report(report: dict[str, Any]) -> dict[str, Any]:
    fanout = await fan_out_report(report)
    synthesis = await synthesize_report(report, fanout)
    return {
        "report": report,
        "fanout": fanout,
        "synthesis": synthesis,
    }


async def process_batch(reports_df: pd.DataFrame) -> list[dict[str, Any]]:
    report_dicts = reports_df.to_dict(orient="records")
    return await asyncio.gather(*(process_one_report(r) for r in report_dicts))

## Run the batch

Make sure your `OPENAI_API_KEY` environment variable is set before running this cell.

In [11]:
results = await process_batch(demo_df)
len(results)

8

## Inspect one example

This is useful for teaching because it shows the intermediate fan-out outputs and the final fan-in synthesis.

In [12]:
example = results[0]
example["report"]["acn"], example["report"]["synopsis"]

('2305728',
 'General aviation pilot reported PABV/BCV Airport has many issues, including pedestrians and vehicles inappropriately using the active taxiways and runways and aircraft not following proper flying procedures and traffic patterns.')

In [13]:
example["fanout"]

{'event_extractor': {'event_summary': 'Pedestrians and vehicles improperly using active taxiways and runways at BCV airport caused unsafe conditions during flight operations.',
  'flight_phase': 'landing and traffic pattern',
  'airport': 'BCV',
  'operation_type': 'general aviation',
  'immediate_outcome': 'unsafe conditions and potential for accident',
  'evidence_quote': '2 ladies walking their 2 dogs on the main taxiway... 2 men walking 3 dogs on Runway 02R... traffic speeding up and down the taxiways... I have had people landing on 20R/L at the same time... I have had people cut under my wings while doing run-up'},
 'delay_driver_classifier': {'primary_driver': 'airport_surface_congestion',
  'secondary_drivers': ['ground_ops', 'security_other'],
  'likely_delay_mechanism': 'Unauthorized pedestrians, vehicles, and animals on active taxiways and runways create safety hazards and operational disruptions, forcing aircraft to delay or alter their movements to avoid conflicts.',
  'con

In [14]:
example["synthesis"]

{'acn': '2305728',
 '_parse_error': True,
 '_raw_text': '```json\n{\n  "acn": 2305728,\n  "airport": "BCV",\n  "date_yyyymm": "202510",\n  "concise_summary": "Pedestrians and vehicles improperly using active taxiways and runways at BCV airport caused unsafe conditions during flight operations, including aircraft not following proper flying procedures and traffic patterns.",\n  "primary_driver": "airport_surface_congestion",\n  "secondary_drivers": [\n    "ground_ops",\n    "security_other"\n  ],\n  "likely_delay": true,\n  "likely_cancellation": false,\n  "likely_gate_return": false,\n  "likely_go_around_or_resequence": true,\n  "disruption_severity": 4,\n  "impact_types": [\n    "taxi_delay",\n    "departure_delay",\n    "arrival_delay",\n    "go_around",\n    "runway_blockage",\n    "passenger_impact"\n  ],\n  "mitigation_bucket": "airport_infrastructure",\n  "recommended_actions": [\n    "Install secure fencing and locked gates to prevent unauthorized pedestrian and vehicle access t

## Flatten the synthesized outputs into a DataFrame

In [15]:
synthesized = [r["synthesis"] for r in results]
df_synth = pd.DataFrame(synthesized)
df_synth

,acn,_parse_error,_raw_text
0,2305728,True,"```json\n{\n ""acn"": 2305728,\n ""airport"": ""B..."
1,2305051,True,"```json\n{\n ""acn"": ""2305051"",\n ""airport"": ..."
2,2303590,True,"```json\n{\n ""acn"": 2303590,\n ""airport"": ""Z..."
3,2301677,True,"```json\n{\n ""acn"": 2301677,\n ""airport"": ""Z..."
4,2301503,True,"```json\n{\n ""acn"": 2301503,\n ""airport"": ""P..."
5,2299197,True,"```json\n{\n ""acn"": 2299197,\n ""airport"": ""A..."
6,2298390,True,"```json\n{\n ""acn"": 2298390,\n ""airport"": ""L..."
7,2294724,True,"```json\n{\n ""acn"": 2294724,\n ""airport"": ""S..."


## Aggregate across reports

Now do a **second-level fan-in**: summarize the whole batch to identify the biggest themes across the set of reports.

In [16]:
BATCH_SUMMARY_SYSTEM = """
You are summarizing a batch of normalized aviation incident records for delay-analysis research.
Return strict JSON only.
"""

async def summarize_batch(df: pd.DataFrame) -> dict[str, Any]:
    records_json = df.to_json(orient="records")
    prompt = f"""
Given these synthesized report records, produce a portfolio-level summary.

Records:
{records_json}

Return JSON with keys:
- top_delay_drivers: list of 5 items with driver and count_or_prevalence_note
- common_operational_impacts: list
- highest_severity_reports: list of ACNs
- delay_research_hypotheses: list of 5 concise hypotheses that could be tested against BTS flight data
- recommended_join_keys: list of fields that would be most useful if joining narrative findings to structured airline data
"""
    response = await client.responses.create(
        model=MODEL,
        instructions=BATCH_SUMMARY_SYSTEM,
        input=prompt,
        temperature=0,
    )
    return json.loads(response.output_text)

batch_summary = await summarize_batch(df_synth)
batch_summary

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

## Simple local summaries without another LLM call

These quick aggregations are useful if you want to show how structured LLM outputs can feed standard analytics.

In [ ]:
driver_counts = (
    df_synth["primary_driver"]
    .value_counts(dropna=False)
    .rename_axis("primary_driver")
    .reset_index(name="count")
)

impact_exploded = (
    df_synth[["acn", "impact_types"]]
    .explode("impact_types")
    .dropna()
)

impact_counts = (
    impact_exploded["impact_types"]
    .value_counts()
    .rename_axis("impact_type")
    .reset_index(name="count")
)

driver_counts, impact_counts.head(10)

## Save outputs

This makes it easy to inspect the LLM-derived structured data outside the notebook.

In [ ]:
out_dir = Path("/mnt/data/asrs_outputs")
out_dir.mkdir(exist_ok=True)

demo_df.to_csv(out_dir / "parsed_reports_demo.csv", index=False)
df_synth.to_csv(out_dir / "synthesized_reports.csv", index=False)

with open(out_dir / "batch_summary.json", "w", encoding="utf-8") as f:
    json.dump(batch_summary, f, indent=2)

print("Wrote files to", out_dir)
list(out_dir.iterdir())

## Teaching takeaway

This notebook demonstrates two levels of fan-out / fan-in:

### Level 1: per-report fan-out / fan-in
One ASRS report is split into four parallel analyses, then merged into one structured synthesis.

### Level 2: batch fan-in
The synthesized report records are then summarized across the whole sample to generate higher-level delay insights and research hypotheses.

That is a practical pattern for turning messy aviation narratives into structured features that can be compared with BTS delay data.